# Chapter 03 — KV Cache & Attention Variants

**Goal:** Understand why KV caching exists, how much memory it costs,
and how architectural variants (GQA, MQA, MLA) reduce that cost.

**Reference model:** LLaMA-3.2-1B  
d_model=2048, 32 Q heads, 8 KV heads, d_head=64, d_ff=8192, 16 layers

**Hardware:** A100-80GB — 1,774 GB/s bandwidth, 312 TFLOPS FP16 (237 TFLOPS effective), ~134 FLOPS/byte ridge point

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No CUDA device — check Remote SSH connection"

device = torch.device('cuda')
props  = torch.cuda.get_device_properties(device)
print(f"GPU      : {props.name}")
print(f"VRAM     : {props.total_memory / 1e9:.1f} GB")
print(f"SM count : {props.multi_processor_count}")
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.version.cuda}")

## 1 — Why KV Cache Exists

### The problem: O(n²) recompute without cache

Autoregressive generation produces one token at a time.  
At step *t* the model needs to attend over all *t* previous tokens.

**Without cache:** every decode step must recompute K and V projections
for the *entire* sequence so far:

- Step 1 → project 1 token  
- Step 2 → project 2 tokens  
- …  
- Step n → project n tokens  
- Total projections ∝ 1 + 2 + … + n = **O(n²)**

**With KV cache:** store K, V from all previous steps. At step *t* we
only project the *new* token's K and V, then append to the cache:

- Total projections = n × 1 = **O(n)**

### Why Q is NOT cached

Q is the *query* for the **current** token — it asks "what should I
attend to?". Each new token needs its own fresh Q. Old Q values are
never reused, so there is nothing to cache.

K and V represent what a token **offers** to future tokens — that
never changes, so we cache them.

## 2 — KV Cache Memory Formula

The formula for KV cache memory per token:

```
kv_bytes = 2 * n_layers * n_kv_heads * d_head * dtype_bytes * seq_len
           ↑                                                  ↑
           K and V                                     tokens cached
```

The factor of 2 is for K *and* V — each is cached separately.

In [ ]:
# ── KV Cache Memory Calculator ──────────────────────────────────────

# LLaMA-3.2-1B parameters
n_layers_1b   = 16
n_kv_heads_1b = 8
d_head         = 64
dtype_bytes    = 2          # FP16

# LLaMA-3.1-70B parameters
n_layers_70b   = 80
n_kv_heads_70b = 8
d_head_70b     = 128

seq_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768, 131072]

def kv_cache_bytes(n_layers, n_kv_heads, d_head, dtype_bytes, seq_len):
    # YOUR CODE
    # Each token stores one K vector and one V vector per layer per KV head.
    # Each vector is d_head elements wide. What's the total across all tokens?
    pass

print("LLaMA-3.2-1B KV cache (FP16):")
for s in seq_lengths:
    size = kv_cache_bytes(n_layers_1b, n_kv_heads_1b, d_head, dtype_bytes, s)
    print(f"  seq={s:>6d}  →  {size / 1e6:>8.1f} MB  ({size / 1e9:.2f} GB)")

print("\nLLaMA-3.1-70B KV cache (FP16):")
for s in seq_lengths:
    size = kv_cache_bytes(n_layers_70b, n_kv_heads_70b, d_head_70b, dtype_bytes, s)
    print(f"  seq={s:>6d}  →  {size / 1e6:>8.1f} MB  ({size / 1e9:.2f} GB)")

## 3 — Measure Actual KV Cache

Theory is nice — let's measure the real thing.  
We load LLaMA-3.2-1B, run inference with and without `use_cache`,
and compare VRAM.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-3.2-1B"
device = "cuda"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.float16, device_map=device
)
model.eval()

prompt = "The meaning of life is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
seq_len = 512

# ── Baseline VRAM (model loaded, no generation) ──
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
baseline_mb = torch.cuda.memory_allocated() / 1e6

# ── Generate WITH cache ──
torch.cuda.reset_peak_memory_stats()
with torch.no_grad():
    out_cached = model.generate(**inputs, max_new_tokens=seq_len, use_cache=True)
torch.cuda.synchronize()
peak_cached_mb = torch.cuda.max_memory_allocated() / 1e6

print(f"Baseline VRAM:         {baseline_mb:.1f} MB")
print(f"Peak with KV cache:    {peak_cached_mb:.1f} MB")
print(f"Difference (measured): {peak_cached_mb - baseline_mb:.1f} MB")

# YOUR CODE: compute expected KV cache size and compare
# Hint: use the formula from Section 2
# expected_kv_mb = kv_cache_bytes(16, 8, 64, 2, seq_len) / 1e6
# print(f"Expected KV cache:     {expected_kv_mb:.1f} MB")
# Note: measured will be larger due to activations, workspace, etc.

## 4 — MHA vs GQA vs MQA Comparison

Three variants of multi-head attention, differing in how many
KV heads they use:

| Variant | KV heads | Description |
|---------|----------|-------------|
| **MHA** (Multi-Head Attention) | = Q heads | Original Transformer. Every Q head has its own K, V head. |
| **GQA** (Grouped Query Attention) | Q heads / group_size | Groups of Q heads share K, V. LLaMA-3 uses this. |
| **MQA** (Multi-Query Attention) | 1 | ALL Q heads share a single K, V. Falcon, PaLM. |

In [ ]:
# ── MHA vs GQA vs MQA KV Cache Comparison ──────────────────────────

# LLaMA-3.1-70B parameters
n_layers   = 80
n_q_heads  = 64
d_head     = 128
dtype_bytes = 2   # FP16
seq_len    = 4096

def kv_cache_gb(n_layers, n_kv_heads, d_head, dtype_bytes, seq_len):
    # YOUR CODE
    # Same formula as Section 2 — return the result in GB (divide by 1e9).
    pass

print(f"LLaMA-3.1-70B KV cache at seq_len={seq_len} (FP16):")
print(f"  MHA (64 KV heads): {kv_cache_gb(n_layers, 64, d_head, dtype_bytes, seq_len):.2f} GB")
print(f"  GQA ( 8 KV heads): {kv_cache_gb(n_layers,  8, d_head, dtype_bytes, seq_len):.2f} GB")
print(f"  MQA ( 1 KV head):  {kv_cache_gb(n_layers,  1, d_head, dtype_bytes, seq_len):.3f} GB")
print()

# YOUR CODE: compute the ratio of MHA cache to GQA cache
# mha_to_gqa_ratio = ???
# print(f"GQA is {mha_to_gqa_ratio:.0f}× smaller than MHA — minimal quality loss.")

## 5 — Multi-Latent Attention (MLA)

### DeepSeek-V2/V3's approach

Instead of caching full K, V for each head, MLA compresses the KV
representation into a **low-rank latent vector** per token:

1. Project KV into a small latent: `c_kv = W_dkv @ x`  (d_latent << n_kv_heads × d_head)
2. Cache only `c_kv` per token per layer
3. At decode time, up-project back: `K = W_uk @ c_kv`, `V = W_uv @ c_kv`

The cache stores `d_latent` values per token instead of `2 × n_kv_heads × d_head`.

DeepSeek-V2 uses d_latent = 512 with 128 Q heads and d_head = 128,
so the compression ratio vs MHA is:
- MHA would cache: 2 × 128 × 128 = 32,768 values per token per layer
- MLA caches: 512 values per token per layer
- Compression: **64×**

In [ ]:
# ── MLA Cache Size Calculator ────────────────────────────────────────

# DeepSeek-V2 parameters (approximate)
n_layers_ds   = 60
d_latent      = 512       # compressed KV dimension
n_q_heads_ds  = 128
d_head_ds     = 128
n_kv_heads_ds = 128       # if it were MHA
dtype_bytes   = 2
seq_len       = 4096

def mla_cache_bytes(n_layers, d_latent, dtype_bytes, seq_len):
    # YOUR CODE
    # MLA compresses K and V into a single latent vector of size d_latent.
    # Unlike GQA, there's no separate K and V — just one compressed representation.
    # What gets stored per token per layer?
    pass

def gqa_cache_bytes(n_layers, n_kv_heads, d_head, dtype_bytes, seq_len):
    # YOUR CODE
    # Same formula as Section 2 — K and V stored separately per KV head.
    pass

mla_gb = mla_cache_bytes(n_layers_ds, d_latent, dtype_bytes, seq_len) / 1e9
gqa_8_gb = gqa_cache_bytes(n_layers_ds, 8, d_head_ds, dtype_bytes, seq_len) / 1e9
mha_gb = gqa_cache_bytes(n_layers_ds, n_kv_heads_ds, d_head_ds, dtype_bytes, seq_len) / 1e9

print(f"DeepSeek-V2 at seq_len={seq_len}:")
print(f"  MHA cache:           {mha_gb:.1f} GB")
print(f"  GQA (8 heads) cache: {gqa_8_gb:.2f} GB")
print(f"  MLA cache:           {mla_gb:.3f} GB")
print(f"  MLA compression vs MHA: {mha_gb / mla_gb:.0f}×")

## 6 — RoPE: Rotary Position Embedding

### How RoPE encodes position

RoPE (Su et al. 2021) encodes position by **rotating** the Q and K vectors.

Key properties:
- **Zero memory cost** — rotations are computed on-the-fly, nothing stored
- **Relative position** — the dot product Q·K depends only on the
  *difference* in positions, not absolute positions
- Works by pairing up dimensions and applying 2D rotations with
  frequency determined by the dimension index

For dimension pair (i, i+1) at position p:
```
θ_i = 10000^(-2i/d)
rotation angle = p × θ_i
```

Applied as:
```
x_i'   = x_i × cos(pθ) - x_{i+1} × sin(pθ)
x_{i+1}' = x_i × sin(pθ) + x_{i+1} × cos(pθ)
```

In [ ]:
import torch
import matplotlib.pyplot as plt

d_head = 64
base = 10000.0

def apply_rope(x, position):
    """Apply RoPE to a vector x at given position."""
    d = x.shape[-1]
    # YOUR CODE
    # Hint:
    # 1. Create frequency vector: theta_i = base^(-2i/d) for i in range(d//2)
    # 2. Compute angles: position * theta_i
    # 3. Pair up dimensions and apply rotation:
    #    x_rot[..., 0::2] = x[..., 0::2] * cos(angles) - x[..., 1::2] * sin(angles)
    #    x_rot[..., 1::2] = x[..., 0::2] * sin(angles) + x[..., 1::2] * cos(angles)
    pass

# Test: dot product should depend on RELATIVE position
q = torch.randn(d_head)
k = torch.randn(d_head)

# Same relative distance (5) at different absolute positions
dot_0_5 = torch.dot(apply_rope(q, 0), apply_rope(k, 5))
dot_100_105 = torch.dot(apply_rope(q, 100), apply_rope(k, 105))
print(f"dot(q@pos=0, k@pos=5)     = {dot_0_5.item():.4f}")
print(f"dot(q@pos=100, k@pos=105) = {dot_100_105.item():.4f}")
print(f"These should be equal (same relative distance of 5)")

# Plot: dot product decays with relative distance
distances = range(0, 200)
dots = [torch.dot(apply_rope(q, 0), apply_rope(k, d)).item() for d in distances]
plt.figure(figsize=(10, 4))
plt.plot(distances, dots)
plt.xlabel("Relative position distance")
plt.ylabel("Dot product (unnormalized attention)")
plt.title("RoPE: dot product vs relative position")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7 — Max Context Length Calculator

Given a fixed VRAM budget, how long a context can we serve?

```
VRAM_available = VRAM_total - model_size - overhead
max_seq_len    = VRAM_available / kv_bytes_per_token
```

This is why GQA matters: fewer KV heads → smaller per-token cost → longer context.

In [ ]:
# ── Max Context Length Calculator ────────────────────────────────────

vram_total_gb = 80.0    # A100
overhead_gb   = 2.0     # CUDA workspace, activations, etc.

# LLaMA-3.1-70B (FP16)
model_size_gb = 140.0   # ~70B params × 2 bytes (needs multi-GPU, but let's calculate)
n_layers      = 80
d_head        = 128
dtype_bytes   = 2

def kv_bytes_per_token(n_layers, n_kv_heads, d_head, dtype_bytes):
    # YOUR CODE
    # Same as the KV cache formula but for a single token — no seq_len factor.
    pass

def max_context(vram_gb, model_gb, overhead_gb, bytes_per_token):
    # YOUR CODE
    # After subtracting model and overhead from VRAM, how many tokens fit?
    pass

# Compare GQA vs MHA for max context
# (70B needs multi-GPU — use 4× A100 = 320 GB for the calculation)
for n_kv in [64, 8, 1]:
    bpt = kv_bytes_per_token(n_layers, n_kv, d_head, dtype_bytes)
    label = {64: "MHA", 8: "GQA", 1: "MQA"}[n_kv]
    max_ctx = max_context(320.0, model_size_gb, overhead_gb * 4, bpt)
    print(f"  {label:3s} ({n_kv:2d} KV heads): {bpt:>10,.0f} bytes/token → max context = {max_ctx:>10,.0f} tokens")

print("\nGQA enables ~8× longer context than MHA for the same VRAM budget.")

## 8 — Decode Attention Bottleneck

### Why decode-step attention is ALWAYS memory-bound

During decode, we generate one token at a time. The attention computation:

1. **Q/K/V projection** — project 1 token: tiny matmul (1 × d_model) @ (d_model × d_head)
2. **Score computation** — Q (1 × d_head) @ K_cache^T (d_head × seq_len) → (1 × seq_len)
3. **Read entire V cache** — scores (1 × seq_len) @ V_cache (seq_len × d_head) → (1 × d_head)

The bottleneck: we must **read the entire KV cache** from HBM for
every single generated token, but only do O(seq_len × d_head) FLOPs.

Arithmetic intensity = FLOPs / bytes_read ≈ 1 FLOP/byte — deeply
memory-bound, far left of the roofline.

In [ ]:
# ── Decode Attention: Arithmetic Intensity Analysis ────────────────

# LLaMA-3.2-1B during decode (1 token at a time)
n_layers   = 16
n_q_heads  = 32
n_kv_heads = 8
d_head     = 64
dtype_bytes = 2

seq_lengths = [128, 512, 2048, 8192, 32768]

ridge_point = 134  # A100 FLOPS/byte

def decode_attn_bytes(n_kv_heads, d_head, dtype_bytes, seq_len):
    """Bytes read from HBM for decode attention (one layer)."""
    # YOUR CODE
    # For each KV head: the entire K cache and V cache must be read from HBM.
    # Each cache stores seq_len vectors of size d_head at dtype_bytes each.
    return None

def decode_attn_flops(n_q_heads, d_head, seq_len):
    """FLOPs for decode attention (one layer)."""
    # YOUR CODE
    # Each Q head computes: Q @ K^T (dot products) and scores @ V (weighted sum).
    # Q is (1, d_head). K_cache is (seq_len, d_head). V_cache is (seq_len, d_head).
    # Count each multiply-accumulate as 2 FLOPs.
    return None

print("Decode attention arithmetic intensity (per layer):")
print(f"{'seq_len':>8s}  {'KV read (KB)':>12s}  {'FLOPs':>10s}  {'AI (FLOP/byte)':>15s}  {'Bound':>10s}")
print("-" * 65)

for s in seq_lengths:
    kv_bytes = decode_attn_bytes(n_kv_heads, d_head, dtype_bytes, s)
    flops = decode_attn_flops(n_q_heads, d_head, s)
    ai = flops / kv_bytes
    bound = "MEM-BOUND" if ai < ridge_point else "COMPUTE-BOUND"
    print(f"{s:>8d}  {kv_bytes/1024:>12.1f}  {flops:>10,d}  {ai:>15.1f}  {bound:>10s}")

print(f"\nA100 ridge point: {ridge_point} FLOP/byte")
print("Notice: AI is CONSTANT — both bytes and FLOPs scale linearly with seq_len.")
print("With GQA (fewer KV heads), AI is HIGHER than MHA (same FLOPs, fewer bytes read).")
print("But still far below the ridge point — decode attention is always memory-bound.")

## Revision Notes

*Fill this in yourself after running all sections. Writing the summary is part of the learning.*

---

**KV cache formula:**
- KV cache bytes = ____________________________
- LLaMA-3.2-1B at seq=4096: ____ MB
- LLaMA-3.1-70B (GQA-8) at seq=4096: ____ GB

**Attention variants (70B at seq=4096):**
- MHA (64 KV heads): ____ GB
- GQA ( 8 KV heads): ____ GB  →  ____× smaller than MHA
- MQA ( 1 KV head):  ____ GB
- MLA: ____× compression vs MHA by storing a ____-dimensional latent instead of full K, V

**Decode attention:**
- Arithmetic intensity (LLaMA-3.2-1B GQA): ____ FLOPS/byte — region: ________
- Why is AI constant as seq_len grows? ____________________________
- Does GQA change AI compared to MHA? ____ — why? ____________________________

**RoPE:**
- Memory cost: ____
- Dot product depends on ____ position, not ____ position

**Key insight (in your own words):**
*(Why does longer context hurt inference speed, and what's the fix?)*


**What surprised me:**


---
*Next: Chapter 4 — Flash Attention*

## Knowledge Check

Test yourself before moving on. Answer from memory and reasoning — no peeking at cells above.

- **Calculation questions** use LLaMA-3.2-1B (d_model=2048, 32 Q heads, 8 KV heads, d_head=64, 16 layers) and LLaMA-3.1-70B (64 Q heads, 8 KV heads, d_head=128, 80 layers)
- **Conceptual questions** test whether you understand *why* the KV cache and attention variants matter
- **Trap questions** catch common misconceptions about memory, compute, and caching

Fill in each `None  # YOUR ANSWER` below, then run the checker cell to see your score.

In [ ]:
# ============================================================
# KNOWLEDGE CHECK — Chapter 3: KV Cache & Attention Variants
# ============================================================
# Answer each question, then run the next cell to check.

# --- Calculation ---

# Q1: LLaMA-3.2-1B KV cache at seq_len=4096 in FP16.
#     How many MB does it take?
#     (16 layers, 8 KV heads, d_head=64, 2 bytes per value)
q1 = None  # YOUR ANSWER (MB)

# Q2: LLaMA-3.1-70B with MHA (64 KV heads) at seq_len=4096 in FP16.
#     How many GB of KV cache?
#     (80 layers, d_head=128)
q2 = None  # YOUR ANSWER (GB)

# Q3: Same 70B model but with GQA (8 KV heads).
#     By what factor does the KV cache shrink vs MHA?
q3 = None  # YOUR ANSWER (ratio, e.g. 4 means "4× smaller")

# Q4: MLA stores a 512-dimensional latent instead of full K, V per head.
#     For DeepSeek-V2 (128 heads, d_head=128), what is the compression
#     ratio vs MHA? (Compare per-token-per-layer storage)
q4 = None  # YOUR ANSWER (ratio)

# --- Conceptual ---

# Q5: During decode, we generate one token and attend over the full cache.
#     As seq_len doubles, what happens to the arithmetic intensity
#     of the decode attention operation?
q5 = None  # YOUR ANSWER: "doubles", "halves", or "stays the same"

# Q6: GQA reduces KV heads from 64 to 8. Compared to MHA, the decode
#     attention arithmetic intensity...
q6 = None  # YOUR ANSWER: "increases", "decreases", or "stays the same"

# Q7: Which attention variant trades COMPUTE for MEMORY at decode time?
#     (It uses extra FLOPs to reconstruct K, V from a compressed cache)
q7 = None  # YOUR ANSWER: "GQA", "MQA", or "MLA"

# Q8: RoPE encodes position by rotating Q and K vectors.
#     How much EXTRA memory does RoPE add to the KV cache per token?
q8 = None  # YOUR ANSWER (bytes)

# --- Trap / misconception ---

# Q9: Someone says "GQA makes attention faster because it reduces FLOPs."
#     Is this correct?
q9 = None  # YOUR ANSWER: "yes" or "no"

# Q10: LLaMA-3.2-1B model weights in FP16 = 2.5 GB.
#      At what sequence length does the KV cache EXCEED the model weight size?
#      (Use your formula — solve for seq_len where KV cache bytes > 2.5 GB)
q10 = None  # YOUR ANSWER (approximate seq_len — nearest thousand is fine)

In [ ]:
# ============================================================
# RUN THIS CELL TO CHECK YOUR ANSWERS (do not modify)
# ============================================================
def _check(label, got, expected, tol=0.1):
    if got is None:
        print(f"  ✗ {label}: not answered")
        return 0
    if isinstance(expected, str):
        ok = got.strip().lower() == expected.strip().lower()
    else:
        ok = abs(got - expected) / max(abs(expected), 1e-9) <= tol
    print(f"  {'✓' if ok else '✗'} {label}" + ("" if ok else " — try again"))
    return int(ok)

_score = 0
_total = 10
print("Knowledge Check Results:\n")

# Q1: 1B KV cache at seq=4096
_q1_expected = 2 * 16 * 8 * 64 * 2 * 4096 / 1e6
_score += _check("Q1: 1B KV cache at seq=4096 (MB)", q1, _q1_expected)

# Q2: 70B MHA cache at seq=4096
_q2_expected = 2 * 80 * 64 * 128 * 2 * 4096 / 1e9
_score += _check("Q2: 70B MHA cache at seq=4096 (GB)", q2, _q2_expected)

# Q3: MHA to GQA ratio
_score += _check("Q3: MHA/GQA cache ratio", q3, 64 / 8)

# Q4: MLA compression ratio
_mha_per_token = 2 * 128 * 128
_mla_per_token = 512
_score += _check("Q4: MLA compression vs MHA", q4, _mha_per_token / _mla_per_token)

# Q5: AI vs seq_len
_score += _check("Q5: AI when seq_len doubles", q5, "stays the same")

# Q6: GQA effect on AI
_score += _check("Q6: GQA effect on decode AI", q6, "increases")

# Q7: Which variant trades compute for memory
_score += _check("Q7: Compute-for-memory variant", q7, "MLA")

# Q8: RoPE extra memory
_score += _check("Q8: RoPE extra KV cache bytes", q8, 0)

# Q9: GQA reduces FLOPs?
_score += _check("Q9: GQA reduces FLOPs?", q9, "no")

# Q10: seq_len where KV > model weights
_kv_per_token = 2 * 16 * 8 * 64 * 2
_q10_expected = 2.5e9 / _kv_per_token
_score += _check("Q10: seq_len where KV > weights", q10, _q10_expected)

print(f"\nScore: {_score}/{_total}")
if _score == _total:
    print("Perfect — you're ready for Chapter 4!")
elif _score >= 7:
    print("Good — review the questions you missed before moving on.")
else:
    print("Review the notebook sections above and try again.")